# Product Demand Forecasting

## Project Overview

This project forecasts **daily product demand** across multiple products using statistical,
ML, and AutoML approaches. We generate synthetic daily demand data for 3 products over 2 years,
each with different seasonality and trend patterns.

**Models**: Naive baselines, LazyPredict (tabular), FLAML AutoML, StatsForecast.

**Forecast horizon**: 14 days ahead per product.

## Learning Objectives

1. Handle multi-product demand forecasting
2. Understand product-specific seasonality and trend patterns
3. Build lag/rolling features for demand time series
4. Compare naive, ML, and statistical approaches per product
5. Use FLAML and StatsForecast effectively
6. Evaluate with MAE / RMSE / MAPE

## Problem Statement

Given historical daily demand for multiple products, forecast the next 14 days of demand
per product. Accurate demand forecasts reduce stockouts and overstock costs.

## Why This Project Matters

Product demand forecasting is central to supply chain management. Under-forecasting causes
stockouts and lost revenue; over-forecasting leads to excess inventory and waste.

## Dataset Overview

Synthetic dataset:
- 3 products, each with ~730 days (2 years) of daily demand
- Product A: stable with weekly seasonality
- Product B: growing trend with monthly peaks
- Product C: declining with strong weekly pattern

| Column | Description |
|--------|-------------|
| `date` | Date |
| `product` | Product identifier (A, B, C) |
| `demand` | Daily units demanded |

## Dataset Source and License Notes

Synthetically generated for educational purposes.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_DAYS = 730
HORIZON = 14
VAL_DAYS = 14
TEST_DAYS = 14
SEASON_LENGTH = 7
np.random.seed(SEED)
print(f"Config: {N_DAYS} days, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 days, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_DAYS, freq='D')
t = np.arange(N_DAYS)

records = []
# Product A: stable + weekly seasonality
demand_a = 200 + 30 * np.sin(2 * np.pi * t / 7) + np.random.normal(0, 15, N_DAYS)
# Product B: growing + monthly peaks
demand_b = 100 + 0.2 * t + 20 * np.sin(2 * np.pi * t / 30) + np.random.normal(0, 10, N_DAYS)
# Product C: declining + weekly
demand_c = 300 - 0.15 * t + 25 * np.sin(2 * np.pi * t / 7) + np.random.normal(0, 12, N_DAYS)

for i, d in enumerate(dates):
    records.append({'date': d, 'product': 'A', 'demand': max(10, demand_a[i])})
    records.append({'date': d, 'product': 'B', 'demand': max(10, demand_b[i])})
    records.append({'date': d, 'product': 'C', 'demand': max(10, demand_c[i])})

df = pd.DataFrame(records)
df['demand'] = df['demand'].round(0).astype(int)
print(f"Dataset shape: {df.shape}")
print(f"Products: {df['product'].unique()}")
df.head(6)

Dataset shape: (2190, 3)
Products: ['A' 'B' 'C']


,date,product,demand
0,2022-01-01,A,207
1,2022-01-01,B,102
2,2022-01-01,C,298
3,2022-01-02,A,221
4,2022-01-02,B,95
5,2022-01-02,C,305


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0
assert (df['demand'] > 0).all()
assert df.groupby('product').size().nunique() == 1, "Unequal product lengths"
print(f"Each product has {len(df) // 3} days. All checks passed.")

Each product has 730 days. All checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for i, prod in enumerate(['A', 'B', 'C']):
    sub = df[df['product'] == prod]
    axes[i].plot(sub['date'], sub['demand'], linewidth=0.6)
    axes[i].set_title(f'Product {prod} — Daily Demand')
    axes[i].set_ylabel('Demand')
axes[2].set_xlabel('Date')
plt.tight_layout()
plt.savefig('eda_product_demand.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
for prod in ['A', 'B', 'C']:
    sub = df[df['product'] == prod]['demand']
    print(f"Product {prod}: mean={sub.mean():.1f}, std={sub.std():.1f}, min={sub.min()}, max={sub.max()}")

Product A: mean=199.8, std=25.5, min=134, max=275
Product B: mean=174.1, std=45.3, min=72, max=282
Product C: mean=245.6, std=38.7, min=150, max=339


## Train / Validation / Test Split Strategy

Time-aware split per product (no shuffling):
- Train: all days except last 28
- Validation: 14 days
- Test: last 14 days

In [8]:
# We'll forecast Product A as the main demo, then loop all products for StatsForecast
prod_a = df[df['product'] == 'A'].copy().reset_index(drop=True)
train_a = prod_a.iloc[:-(VAL_DAYS + TEST_DAYS)]
val_a = prod_a.iloc[-(VAL_DAYS + TEST_DAYS):-TEST_DAYS]
test_a = prod_a.iloc[-TEST_DAYS:]
print(f"Product A — Train: {len(train_a)}, Val: {len(val_a)}, Test: {len(test_a)}")

Product A — Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing needed — daily integer demand, no missing values. Features next.

In [9]:
print('Data ready for feature engineering.')

Data ready for feature engineering.


## Feature Engineering

In [10]:
def create_features(data, target='demand'):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[target].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[target].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[target].shift(1).rolling(w).std()
    return d

feat_a = create_features(prod_a).dropna().reset_index(drop=True)
feature_cols = [c for c in feat_a.columns if c not in ['date', 'demand', 'product']]
print(f"Features: {len(feature_cols)} columns, {len(feat_a)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:8.2f} | RMSE: {rmse:8.2f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_DAYS, train_a['demand'].iloc[-1])
results.append(calc_metrics(test_a['demand'].values, naive_pred, "Naive (Last Value)"))

# Seasonal naive
src = prod_a['demand'].values
ti = len(prod_a) - TEST_DAYS
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_DAYS]
results.append(calc_metrics(test_a['demand'].values, sn_pred, "Seasonal Naive (7d)"))

Naive (Last Value)             | MAE:    42.50 | RMSE:    47.74 | MAPE: 23.07%
Seasonal Naive (7d)            | MAE:    10.07 | RMSE:    14.96 | MAPE:  5.07%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(feat_a) - TEST_DAYS
cv = ct - VAL_DAYS
X_tr = feat_a.iloc[:cv][feature_cols]
y_tr = feat_a.iloc[:cv]['demand']
X_va = feat_a.iloc[cv:ct][feature_cols]
y_va = feat_a.iloc[cv:ct]['demand']

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                           
KernelRidge                        635.322392 -47.794030  199.242136    0.031441
GaussianProcessRegressor           200.187685 -14.322130  111.649661    0.085812
MLPRegressor                        21.070834  -0.543910   35.441237    0.370859
DummyRegressor                      14.064486  -0.004960   28.593830    0.005763
QuantileRegressor                   14.063916  -0.004917   28.593206    0.077554
RANSACRegressor                     13.900182   0.007678   28.413457    0.077486
OrthogonalMatchingPursuit           11.199655   0.215411   25.264977    0.012858
DecisionTreeRegressor               10.782529   0.247498   24.742964    0.012435
BaggingRegressor                     9.190342   0.369974   22.640040    0.051412
ExtraTreeRegressor                   9.132133   0.374451   22.559445    0.008467


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = feat_a.iloc[ct:][feature_cols]
y_te = feat_a.iloc[ct:]['demand']
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:    13.95 | RMSE:    17.65 | MAPE:  7.07%
FLAML Test (extra_tree)        | MAE:     8.63 | RMSE:    12.42 | MAPE:  4.41%


## StatsForecast — Statistical Forecasting (All Products)

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df[['date', 'product', 'demand']].copy()
sf_df.columns = ['ds', 'unique_id', 'y']

sf_train = sf_df.groupby('unique_id').apply(lambda g: g.iloc[:-TEST_DAYS], include_groups=False).reset_index(level=0)
sf_test = sf_df.groupby('unique_id').apply(lambda g: g.iloc[-TEST_DAYS:], include_groups=False).reset_index(level=0)

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH), SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq='D', n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_DAYS).reset_index()

# Product A results
sf_a = sf_preds[sf_preds['unique_id'] == 'A']
y_test_a = test_a['demand'].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_a.columns:
        results.append(calc_metrics(y_test_a, sf_a[col].values, f"SF {col} (Prod A)"))

print("\nStatsForecast complete for all products.")

SF AutoARIMA (Prod A)          | MAE:     9.15 | RMSE:    12.01 | MAPE:  4.69%
SF AutoETS (Prod A)            | MAE:     7.99 | RMSE:    11.06 | MAPE:  4.08%
SF SeasonalNaive (Prod A)      | MAE:    11.50 | RMSE:    16.40 | MAPE:  5.73%

StatsForecast complete for all products.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results (Product A):")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results (Product A):
                    model       MAE      RMSE      MAPE
      SF AutoETS (Prod A)  7.993955 11.063196  4.082943
  FLAML Test (extra_tree)  8.629540 12.417244  4.410680
    SF AutoARIMA (Prod A)  9.151906 12.011937  4.693550
      Seasonal Naive (7d) 10.071429 14.959469  5.066001
SF SeasonalNaive (Prod A) 11.500000 16.403397  5.734837
       FLAML (extra_tree) 13.945855 17.649199  7.072976
       Naive (Last Value) 42.500000 47.735132 23.065951

Top 3:
                  model      MAE      RMSE     MAPE
    SF AutoETS (Prod A) 7.993955 11.063196 4.082943
FLAML Test (extra_tree) 8.629540 12.417244 4.410680
  SF AutoARIMA (Prod A) 9.151906 12.011937 4.693550


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_a['date'].values, test_a['demand'].values, 'ko-', label='Actual', ms=4)
ax.plot(test_a['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_a.columns:
        ax.plot(test_a['date'].values[:len(sf_a)], sf_a[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Product A — Forecast vs Actual')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test_a['demand'].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -3.06, Std: 12.04


## Interpretation and Business Insight

- Product A has stable demand — simple models work well
- Product B's growth trend requires trend-aware models
- Product C's decline needs monitoring for discontinuation decisions
- Weekly seasonality is the dominant pattern across all products

## Limitations

1. Synthetic data — real demand has promotions, stockouts, substitution effects
2. Independent product forecasting — ignores cross-product cannibalization
3. No external features (price, advertising, competitor actions)
4. Point forecasts only — no prediction intervals

## How to Improve This Project

1. Add promotion/price features
2. Use hierarchical forecasting across product categories
3. Add prediction intervals for safety stock calculations
4. Model cross-product demand correlations
5. Use Chronos-Bolt for zero-shot demand forecasting

## Production Considerations

- Daily automated retraining
- Alert on demand anomalies
- Integration with inventory management systems
- A/B test forecast-driven vs. rule-based ordering

## Common Mistakes

1. Not handling zero-demand days (intermittent demand)
2. Ignoring stockout censoring — low sales ≠ low demand during stockouts
3. Using shuffled CV on time-series data
4. Forecasting at wrong granularity

## Mini Challenge / Exercises

1. Forecast all 3 products and compare which is easiest/hardest
2. Add a price feature and see if demand elasticity is captured
3. Implement intermittent demand (Croston's method) for a slow-moving product
4. Build a simple ensemble of FLAML + AutoARIMA

## Final Summary / Key Takeaways

1. Multi-product demand forecasting requires product-specific analysis
2. Weekly seasonality dominates daily product demand patterns
3. FLAML with lag features works well for stable-demand products
4. StatsForecast provides interpretable baselines across all products
5. Real-world demand forecasting needs external features and censoring awareness

In [18]:
import json
metrics = {
    'project': 'Product Demand Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Product Demand Forecasting — Complete ===")

Metrics saved.

=== Product Demand Forecasting — Complete ===
